# LUTM-6 · simulate

Choose a backend, a raw program and an input. All three backends use the same physical machine and fixed input/readout convention. `scalar` has self-extending tapes; `numpy` and `cuda` use the explicit padded bounds below. Run cells in order; restart the kernel before reinitializing CUDA.


In [ ]:
from itertools import product
import numpy as np

from machine import Machine, encode_input
from programs import TASKS, load_program
from utils import SimulationConfig, pack_programs, Dataset
from numpy_backend import NumpySimulator

backend = "numpy"  # "scalar", "numpy", or "cuda"
if backend == "cuda":
    from taichi_backend import initialize_cuda, TaichiSimulator
    initialize_cuda()


In [ ]:
# Use a known literal, or replace program with any string over B,0,1,#.
task_name = "bit_not"
program = load_program(task_name)
input_bits = "00101101"

# P and X both start at coordinate zero; this X encoding ignores the program.
print("program length:", len(program))
print("input:", repr(input_bits), "  E(x):", encode_input(input_bits))


In [ ]:
# Edit these for bounded NumPy/CUDA execution. Bounds are inclusive.
config = SimulationConfig(
    max_program_length=max(128, len(program)),
    program_left_budget=8,
    program_right_budget=max(136, len(program) + 1),
    data_left_budget=32,
    data_right_budget=255,
    t_max=1_000_000,
)
scalar_timeout = 10_000_000
capacity = 512


In [ ]:
if backend == "scalar":
    scalar = Machine(program, input_bits)
    result = scalar.run(scalar_timeout)
    print(result)
elif backend in ("numpy", "cuda"):
    simulator = (NumpySimulator(config, capacity) if backend == "numpy"
                 else TaichiSimulator(config, capacity))
    values, lengths = pack_programs([program], config.max_program_length)
    result = simulator.simulate(values, lengths, input_bits, tapes=True)
    print(result.row())
else:
    raise ValueError("choose scalar, numpy or cuda")
print("expected for the selected registered task:", repr(TASKS[task_name](input_bits)))


## Inspect a tape window

`B` is a physical blank, not a missing predicted bit: `1B` decodes to zero. The data head at halt is the start of readout; no head-reset requirement is imposed.


In [ ]:
from machine import SYMBOLS

window_left, window_right = -4, 25
if backend == "scalar":
    print("X:", "".join(scalar.d.get(i, "B") for i in range(window_left, window_right + 1)))
else:
    lo = max(window_left, -config.data_left_budget)
    hi = min(window_right, config.data_right_budget)
    print(f"X [{lo}, {hi}]:", "".join(SYMBOLS[int(v)] for v in result.data_tape[0, lo + config.data_left_budget:hi + config.data_left_budget + 1]))


## Optional batch of raw programs

Independent executions on one input. Explicit lengths preserve active `B` genes; inactive array padding is also `B`.


In [ ]:
if backend != "scalar":
    candidates = ["", load_program("bit_not"), load_program("parity"), "01#1B"]
    values, lengths = pack_programs(candidates, config.max_program_length)
    batch = simulator.simulate(values, lengths, input_bits)
    for i, candidate in enumerate(candidates):
        print(repr(candidate), batch.row(i))


## Optional exhaustive known-task check (unbounded scalar)

Select tasks and input-length range below, then set `run_known_checks=True`. Square is a very slow compiled witness: use `tests/validate_known.py --physical` for selected CPU-accelerated physical runs instead of a large scalar sweep. These examples establish finite correctness checks, not arbitrary-length proofs.


In [ ]:
run_known_checks = False
selected_tasks = ["identity", "bit_not", "parity", "append0", "append1"]
min_length, max_length = 0, 3
check_timeout = 10_000_000

if run_known_checks:
    if not 0 <= min_length <= max_length:
        raise ValueError("require 0 <= min_length <= max_length")
    for name in selected_tasks:
        literal = load_program(name)
        for length in range(min_length, max_length + 1):
            for word in product("01", repeat=length):
                x = "".join(word)
                outcome = Machine(literal, x).run(check_timeout)
                if not outcome["halted"] or outcome["output"] != TASKS[name](x):
                    raise RuntimeError((name, x, outcome))
                print(name, repr(x), "->", repr(outcome["output"]), "T=", outcome["T"])
